In [2]:
import time

class SudokuCSP:
    def __init__(self, boardStr):
        # List all variables as (row, col) tuples
        self.variables = [(r, c) for r in range(9) for c in range(9)]
        
        # Initialize domains (1-9) for empty cells (0), fixed value for others
        self.domains = {
            v: list(range(1, 10)) if boardStr[i] == '0' else [int(boardStr[i])] 
            for i, v in enumerate(self.variables)
        }
        
        self.constraints = self.generateConstraints()
        self.backtrackCalls = 0
        self.backtrackFailures = 0

    def generateConstraints(self):
        constraints = {}
        for r, c in self.variables:
            neighbors = set()
            for i in range(9):
                if i != c:
                    neighbors.add((r, i))
                if i != r:
                    neighbors.add((i, c))
            
            br, bc = 3 * (r // 3), 3 * (c // 3)
            for i in range(br, br + 3):
                for j in range(bc, bc + 3):
                    if (i, j) != (r, c):
                        neighbors.add((i, j))
            constraints[(r, c)] = neighbors
        return constraints

    def ac3(self):
        queue = [(xi, xj) for xi in self.variables for xj in self.constraints[xi]]
        while queue:
            xi, xj = queue.pop(0)
            if self.revise(xi, xj):
                if not self.domains[xi]:
                    return False
                for xk in self.constraints[xi]:
                    if xk != xj:
                        queue.append((xk, xi))
        return True

    def revise(self, xi, xj):
        revised = False
        if len(self.domains[xj]) == 1:
            valJ = self.domains[xj][0]
            if valJ in self.domains[xi]:
                self.domains[xi].remove(valJ)
                revised = True
        return revised

    def solve(self):
        if not self.ac3():
            return None
        
        initialAssignment = {
            v: self.domains[v][0] 
            for v in self.variables if len(self.domains[v]) == 1
        }
        return self.backtrack(initialAssignment)

    def backtrack(self, assignment):
        self.backtrackCalls += 1
        
        if len(assignment) == 81:
            return assignment

        var = self.selectUnassignedVariable(assignment)
        
        for value in list(self.domains[var]):
            if self.isConsistent(var, value, assignment):
                assignment[var] = value
                savedDomains = {v: list(d) for v, d in self.domains.items()}
                
                if self.forwardCheck(var, value):
                    result = self.backtrack(assignment)
                    if result:
                        return result
                
                self.domains = savedDomains
                del assignment[var]
        
        self.backtrackFailures += 1
        return None

    def selectUnassignedVariable(self, assignment):
        unassigned = [v for v in self.variables if v not in assignment]
        return min(unassigned, key=lambda v: len(self.domains[v]))

    def forwardCheck(self, var, value):
        for neighbor in self.constraints[var]:
            if value in self.domains[neighbor]:
                self.domains[neighbor].remove(value)
                if not self.domains[neighbor]:
                    return False
        return True

    def isConsistent(self, var, value, assignment):
        for neighbor in self.constraints[var]:
            if neighbor in assignment and assignment[neighbor] == value:
                return False
        return True


def printPrettySudoku(assignment):
    if not assignment:
        print(">> No consistent solution possible for this board.")
        return

    for r in range(9):
        if r > 0 and r % 3 == 0:
            print("------+-------+------")
        
        rowStr = ""
        for c in range(9):
            if c > 0 and c % 3 == 0:
                rowStr += "| "
            val = assignment.get((r, c), ".")
            rowStr += f"{val} "
        print(rowStr.strip())



def read_board_from_file(filename):
    boardStr = ""
    
    with open(filename, 'r') as file:
        lines = file.readlines()
        
        if len(lines) != 9:
            raise ValueError("File must contain exactly 9 lines.")
        
        for line in lines:
            line = line.strip()
            
            if len(line) != 9 or not line.isdigit():
                raise ValueError("Each line must contain exactly 9 digits (0-9).")
            
            boardStr += line
    
    return boardStr



filename = input("Enter Sudoku file name (e.g., easy.txt): ")

try:
    boardStr = read_board_from_file(filename)
    
    print(f"\n{'-'*10} Solving Sudoku from File {'-'*10}")
    
    solver = SudokuCSP(boardStr)
    
    startTime = time.time()
    solution = solver.solve()
    endTime = time.time()
    
    printPrettySudoku(solution)
    print(f"\nBacktrack Calls: {solver.backtrackCalls}")
    print(f"Backtrack Failures: {solver.backtrackFailures}")
    print(f"Execution Time: {endTime - startTime:.4f} seconds")

except Exception as e:
    print("Error:", e)

Enter Sudoku file name (e.g., easy.txt):  hard.txt



---------- Solving Sudoku from File ----------
4 1 7 | 3 6 9 | 8 2 5
6 3 2 | 1 5 8 | 9 4 7
9 5 8 | 7 2 4 | 3 1 6
------+-------+------
8 2 5 | 4 3 7 | 1 6 9
7 9 1 | 5 8 6 | 4 3 2
3 4 6 | 9 1 2 | 7 5 8
------+-------+------
2 8 9 | 6 4 3 | 5 7 1
5 7 3 | 2 9 1 | 6 8 4
1 6 4 | 8 7 5 | 2 9 3

Backtrack Calls: 420
Backtrack Failures: 355
Execution Time: 0.0211 seconds
